# Reproducing the neural set-propagation experiment

This notebook reproduces computations for a fixed random feedforward network acting on the input set
\[
X_0=[0,1]^2 \subset \mathbb{R}^2.
\]

The network has:
- input dimension $2$,
- hidden width $30$,
- output dimension $2$,
- five affine layers in total (linear maps plus bias),
- componentwise $\tanh$ activations after every affine layer except the last.

So the map is
\[
F = T_5 \circ \tanh \circ T_4 \circ \tanh \circ T_3 \circ \tanh \circ T_2 \circ \tanh \circ T_1.
\]

This notebook generates interval-arithmetic (IA) figures:
1. the input square and a dense-sampling approximation of the image $F(X_0)$,
2. the interval-arithmetic enclosure overlaid with the sampled image,
3. the corresponding intervalNets interval propagation enclosure.

This notebook now documents interval-arithmetic propagation only. The word "affine" above refers only to neural-network linear layers plus bias.

## Imports and configuration

We use the same fixed random seed and network-generation procedure as in the previous computation.


In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Fixed random seed and network width
seed = 3
width = 30
rng = np.random.default_rng(seed)

# Directory for saved figures
outdir = Path("neural_set_propagation_outputs")
outdir.mkdir(exist_ok=True)

print(f"seed = {seed}")
print(f"hidden width = {width}")
print(f"figure output directory = {outdir.resolve()}")


## Build the random network

The dimensions are
\[
2 \to 30 \to 30 \to 30 \to 30 \to 2.
\]

Each affine layer has the form $T_i(x)=W_i x + b_i$.  
The same random network is used in all four plots.


In [ ]:
dims = [2, width, width, width, width, 2]

Ws = []
bs = []
for din, dout in zip(dims[:-1], dims[1:]):
    scale = 0.7 / math.sqrt(din)
    W = rng.normal(0.0, scale, size=(dout, din))
    b = rng.normal(0.0, 0.15, size=(dout,))
    Ws.append(W)
    bs.append(b)

print("Network architecture:")
for i, (W, b) in enumerate(zip(Ws, bs), start=1):
    print(f"T{i}: W shape = {W.shape}, b shape = {b.shape}")


## Forward map on point samples

This is the standard pointwise forward evaluation of the network.  
We use it to generate a dense-sampling approximation of the image $F(X_0)$.


In [ ]:
def forward_points(X: np.ndarray) -> np.ndarray:
    """Forward evaluation on a batch of input points of shape (N, 2)."""
    A = X
    for i, (W, b) in enumerate(zip(Ws, bs)):
        A = A @ W.T + b
        if i < len(Ws) - 1:
            A = np.tanh(A)
    return A


## Interval arithmetic (IA) propagation

For an interval box $[\ell, u]$, an affine map is propagated by splitting each matrix into positive and negative parts:
\[
W = W_+ + W_-,
\qquad
W_+ = \max(W,0),\quad W_- = \min(W,0).
\]
Then
\[
[\ell',u'] = [W_+\ell + W_- u + b,\; W_+ u + W_- \ell + b].
\]

Since $\tanh$ is monotone increasing, it is applied coordinatewise to the endpoints.


In [ ]:
def ia_propagate(lo: np.ndarray, hi: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Propagate an interval box through the network using interval arithmetic."""
    for i, (W, b) in enumerate(zip(Ws, bs)):
        W_pos = np.maximum(W, 0.0)
        W_neg = np.minimum(W, 0.0)

        new_lo = W_pos @ lo + W_neg @ hi + b
        new_hi = W_pos @ hi + W_neg @ lo + b
        lo, hi = new_lo, new_hi

        if i < len(Ws) - 1:
            lo = np.tanh(lo)
            hi = np.tanh(hi)

    return lo, hi


## Approximate the exact image by dense sampling

We sample the input square $X_0=[0,1]^2$ on a dense $350 \times 350$ grid and push those points through the network.  
This gives a high-resolution approximation of the true image $F(X_0)$.


In [ ]:
# Dense grid on the input square
n_grid = 350
xs = np.linspace(0.0, 1.0, n_grid)
ys = np.linspace(0.0, 1.0, n_grid)
X0_grid = np.stack(np.meshgrid(xs, ys), axis=-1).reshape(-1, 2)

# Approximate exact image
Y_exact = forward_points(X0_grid)

# Input square polygon
X0_square = np.array(
    [
        [0.0, 0.0],
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
    ],
    dtype=float,
)

print("Approximate exact image bounds from dense sampling:")
print(f"x in [{Y_exact[:,0].min():.6f}, {Y_exact[:,0].max():.6f}]")
print(f"y in [{Y_exact[:,1].min():.6f}, {Y_exact[:,1].max():.6f}]")


## Compute the IA propagated set

In [ ]:
# IA output box
ia_lo, ia_hi = ia_propagate(np.array([0.0, 0.0]), np.array([1.0, 1.0]))
IA_box = np.array(
    [
        [ia_lo[0], ia_lo[1]],
        [ia_hi[0], ia_lo[1]],
        [ia_hi[0], ia_hi[1]],
        [ia_lo[0], ia_hi[1]],
    ],
    dtype=float,
)

print("IA output box:")
print(f"x in [{ia_lo[0]:.6f}, {ia_hi[0]:.6f}]")
print(f"y in [{ia_lo[1]:.6f}, {ia_hi[1]:.6f}]")

## Plot 1: input square and approximate exact image

In [ ]:
fig1, ax1 = plt.subplots(figsize=(7, 7))

sq_closed = close_poly(X0_square)
ax1.fill(X0_square[:, 0], X0_square[:, 1], alpha=0.20, label=r"input $X_0=[0,1]^2$")
ax1.plot(sq_closed[:, 0], sq_closed[:, 1])

ax1.scatter(
    Y_exact[:, 0],
    Y_exact[:, 1],
    s=1,
    alpha=0.25,
    label=r"approximate exact image $F(X_0)$",
)

xlim1, ylim1 = combined_limits([X0_square, Y_exact])
style_axes(ax1, xlim1, ylim1, "Input square and approximate exact image of the random network")
ax1.legend()

path1 = outdir / "plot_1_exact_image.png"
fig1.savefig(path1, dpi=220, bbox_inches="tight")
plt.show()

print(path1.resolve())


## Plot 2: IA enclosure and approximate exact image

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 7))

ia_closed = close_poly(IA_box)
ax2.fill(IA_box[:, 0], IA_box[:, 1], alpha=0.25, label="IA enclosure")
ax2.plot(ia_closed[:, 0], ia_closed[:, 1])

ax2.scatter(
    Y_exact[:, 0],
    Y_exact[:, 1],
    s=1,
    alpha=0.25,
    label=r"approximate exact image $F(X_0)$",
)

xlim2, ylim2 = combined_limits([IA_box, Y_exact])
style_axes(ax2, xlim2, ylim2, "Interval arithmetic propagation vs. approximate exact image")
ax2.legend()

path2 = outdir / "plot_2_ia.png"
fig2.savefig(path2, dpi=220, bbox_inches="tight")
plt.show()

print(path2.resolve())


## Summary

Running this notebook from top to bottom reproduces:
- the same fixed random network,
- the same dense-sampling approximation of the image of $X_0=[0,1]^2$,
- the same IA propagation,
- and the same intervalNets IA comparison figures.

## Repeat the enclosure with intervalNets IA code

This section reproduces the same interval enclosure for the **same random network** using `intervalnets` (`IntervalTensor`).

In [ ]:
import torch
from torch import nn

from intervalnets import IntervalTensor, interval_forward

In [ ]:
# Build the identical network in torch using the same sampled Ws, bs
torch.set_default_dtype(torch.float64)

layers = []
for i, (W_np, b_np) in enumerate(zip(Ws, bs)):
    W_t = torch.tensor(W_np, dtype=torch.float64)
    b_t = torch.tensor(b_np, dtype=torch.float64)
    lin = nn.Linear(W_t.shape[1], W_t.shape[0], bias=True, dtype=torch.float64)
    with torch.no_grad():
        lin.weight.copy_(W_t)
        lin.bias.copy_(b_t)
    layers.append(lin)
    if i < len(Ws) - 1:
        layers.append(nn.Tanh())

model_torch = nn.Sequential(*layers).eval()

# intervalNets IA propagation on X0 = [0,1]^2
X0_interval = IntervalTensor.from_bounds([0.0, 0.0], [1.0, 1.0])
Y_ia_intervalnets = interval_forward(model_torch, X0_interval, enclosure_mode="box")
ia2_lo = np.array(Y_ia_intervalnets.lower, dtype=float)
ia2_hi = np.array(Y_ia_intervalnets.upper, dtype=float)

IA2_box = np.array(
    [
        [ia2_lo[0], ia2_lo[1]],
        [ia2_hi[0], ia2_lo[1]],
        [ia2_hi[0], ia2_hi[1]],
        [ia2_lo[0], ia2_hi[1]],
    ],
    dtype=float,
)

print("intervalNets IA output box:")
print(f"x in [{ia2_lo[0]:.6f}, {ia2_hi[0]:.6f}]")
print(f"y in [{ia2_lo[1]:.6f}, {ia2_hi[1]:.6f}]")

## Plot 5: (intervalNets) input square and approximate exact image


In [ ]:
fig5, ax5 = plt.subplots(figsize=(7, 7))

sq_closed = close_poly(X0_square)
ax5.fill(X0_square[:, 0], X0_square[:, 1], alpha=0.20, label=r"input $X_0=[0,1]^2$")
ax5.plot(sq_closed[:, 0], sq_closed[:, 1])

ax5.scatter(
    Y_exact[:, 0],
    Y_exact[:, 1],
    s=1,
    alpha=0.25,
    label=r"approximate exact image $F(X_0)$",
)

xlim5, ylim5 = combined_limits([X0_square, Y_exact])
style_axes(ax5, xlim5, ylim5, "(intervalNets run) Input square and approximate exact image")
ax5.legend()

path5 = outdir / "plot_5_intervalnets_exact_image.png"
fig5.savefig(path5, dpi=220, bbox_inches="tight")
plt.show()

print(path5.resolve())


## Plot 6: (intervalNets IA) enclosure and approximate exact image


In [ ]:
fig6, ax6 = plt.subplots(figsize=(7, 7))

ia2_closed = close_poly(IA2_box)
ax6.fill(IA2_box[:, 0], IA2_box[:, 1], alpha=0.25, label="intervalNets IA enclosure")
ax6.plot(ia2_closed[:, 0], ia2_closed[:, 1])

ax6.scatter(
    Y_exact[:, 0],
    Y_exact[:, 1],
    s=1,
    alpha=0.25,
    label=r"approximate exact image $F(X_0)$",
)

xlim6, ylim6 = combined_limits([IA2_box, Y_exact])
style_axes(ax6, xlim6, ylim6, "intervalNets interval propagation vs. approximate exact image")
ax6.legend()

path6 = outdir / "plot_6_intervalnets_ia.png"
fig6.savefig(path6, dpi=220, bbox_inches="tight")
plt.show()

print(path6.resolve())
